# Model-Ready MNIST Pipeline Template

This notebook is organized into clear sections.

Goal:
- Complete data processing and feature preparation
- Keep model training/evaluation modular
- Let you plug in any model (KNN, Logistic, Naive Bayes, etc.) with minimal changes

## Section 1 - Setup and Imports
These cells configure paths and import project utilities.

In [ ]:
from pathlib import Path
import sys
import json
from dataclasses import dataclass
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    for parent in [project_root, *project_root.parents]:
        if (parent / "src").exists():
            project_root = parent
            break

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.features.mnist_features import (
    load_mnist,
    select_binary_classes,
    split_data,
    normalize_data,
    build_features,
    class_distribution,
)

from src.tests.evaluations import (
    confusion_matrix_from_predictions,
    macro_precision_recall_f1,
    accuracy_from_confusion,
    roc_curve_binary,
    precision_recall_curve_binary,
    auc_trapezoid,
    average_precision_binary,
)

In [ ]:
@dataclass
class Config:
    random_state: int = 42
    data_path: str = str((project_root / 'data' / 'mnist.npz').resolve())

    class_a: int = 0
    class_b: int = 1

    test_size: float = 0.2
    val_size: float = 0.2

    feature_mode: str = 'flatten'  # flatten | pca | hog
    pca_components: float = 0.95

cfg = Config()
cfg.output_dir.mkdir(parents=True, exist_ok=True)

print('Data path:', cfg.data_path)
print('Output dir:', cfg.output_dir)
print('Classes:', (cfg.class_a, cfg.class_b))
print('Feature mode:', cfg.feature_mode)

Data path: C:\Users\Ahmed Fahmy\Downloads\Projects\ML\Mnist-Classification\data\mnist.npz
Output dir: C:\Users\Ahmed Fahmy\Downloads\Projects\ML\Mnist-Classification\reports\model_ready_pipeline
Classes: (0, 1)
Feature mode: flatten


## Section 2 - Data Loading and Preprocessing
This section loads MNIST, filters classes, creates train/val/test splits, and normalizes pixels.

In [7]:
t0 = perf_counter()

x_all, y_all = load_mnist(cfg.data_path)
x_binary, y_binary = select_binary_classes(x_all, y_all, cfg.class_a, cfg.class_b)

x_train, y_train, x_val, y_val, x_test, y_test = split_data(
    x_binary,
    y_binary,
    test_size=cfg.test_size,
    val_size=cfg.val_size,
    random_state=cfg.random_state,
)

x_train = normalize_data(x_train)
x_val = normalize_data(x_val)
x_test = normalize_data(x_test)

prep_seconds = perf_counter() - t0

print('Prep time (s):', round(prep_seconds, 3))
print('x_train:', x_train.shape, 'x_val:', x_val.shape, 'x_test:', x_test.shape)
print('y_train:', y_train.shape, 'y_val:', y_val.shape, 'y_test:', y_test.shape)

Prep time (s): 1.036
x_train: (8868, 28, 28) x_val: (2956, 28, 28) x_test: (2956, 28, 28)
y_train: (8868,) y_val: (2956,) y_test: (2956,)


In [ ]:
dist_all = class_distribution(y_binary)
dist_train = class_distribution(y_train)
dist_val = class_distribution(y_val)
dist_test = class_distribution(y_test)

print('Class distribution (all):', dist_all)
print('Class distribution (train):', dist_train)
print('Class distribution (val):', dist_val)
print('Class distribution (test):', dist_test)

## Section 3 - Feature Engineering
Build selected features and keep transformations train-fitted only.

In [8]:
x_train_feat, x_val_feat, x_test_feat = build_features(
    mode=cfg.feature_mode,
    x_train=x_train,
    x_val=x_val,
    x_test=x_test,
    pca_components=cfg.pca_components,
    random_state=cfg.random_state,
)

feature_info = {
    'feature_mode': cfg.feature_mode,
    'pca_components': cfg.pca_components if cfg.feature_mode == 'pca' else None,
    'random_state': cfg.random_state,
}

print('Feature shapes:')
print('  train:', x_train_feat.shape)
print('  val  :', x_val_feat.shape)
print('  test :', x_test_feat.shape)
print('Feature info:', feature_info)

Feature shapes:
  train: (8868, 784)
  val  : (2956, 784)
  test : (2956, 784)
Feature info: {'feature_mode': 'flatten', 'pca_components': None, 'random_state': 42}


In [ ]:
# Optional quick visualization for sanity (first 200 train points)
if cfg.feature_mode in ['flatten', 'pca'] and x_train_feat.shape[1] >= 2:
    vis = x_train_feat[:200, :2]
    plt.figure(figsize=(6, 5))
    plt.scatter(vis[:, 0], vis[:, 1], c=y_train[:200], s=10, alpha=0.7)
    plt.title('Quick 2D Feature Sanity View')
    plt.xlabel('f1')
    plt.ylabel('f2')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print('Skipping 2D feature sanity plot for current mode.')

## Section 4 - Plug-In Model Interface
Use a small adapter contract so any model can be inserted with the same pipeline logic.

In [ ]:
class ModelAdapterBase:
    def fit(self, x, y):
        raise NotImplementedError

    def predict(self, x):
        raise NotImplementedError

    def predict_proba(self, x):
        # Optional: return probability for positive class (class_b or class_a depending on your model)
        return None


# TODO: Replace this with your real model adapter.
MODEL_ADAPTER = None

if MODEL_ADAPTER is None:
    print('Model not set yet. Add your model adapter in this cell.')
else:
    print('Model adapter is ready:', type(MODEL_ADAPTER).__name__)

## Section 5 - Shared Evaluation Utilities
These functions stay the same regardless of model type.

In [ ]:
def evaluate_predictions(y_true, y_pred, class_a, class_b):
    labels = [class_a, class_b]
    cm = confusion_matrix_from_predictions(y_true, y_pred, labels)
    precision, recall, f1 = macro_precision_recall_f1(cm)
    accuracy = accuracy_from_confusion(cm)

    return {
        'accuracy': float(accuracy),
        'precision_macro': float(precision),
        'recall_macro': float(recall),
        'f1_macro': float(f1),
        'confusion_matrix': cm,
    }


def evaluate_probabilities(y_true, y_score, positive_label):
    fpr, tpr, _ = roc_curve_binary(y_true, y_score, pos_label=positive_label)
    roc_auc = auc_trapezoid(fpr, tpr)

    precision_curve, recall_curve, _ = precision_recall_curve_binary(
        y_true, y_score, pos_label=positive_label
    )
    avg_precision = average_precision_binary(y_true, y_score, pos_label=positive_label)

    return {
        'fpr': fpr,
        'tpr': tpr,
        'roc_auc': float(roc_auc),
        'precision_curve': precision_curve,
        'recall_curve': recall_curve,
        'avg_precision': float(avg_precision),
    }

## Section 6 - Tests and Validation Checks
Run these checks before training to catch data or preprocessing problems early.

In [ ]:
def run_data_tests():
    assert x_train_feat.shape[0] == y_train.shape[0], 'Train shape mismatch'
    assert x_val_feat.shape[0] == y_val.shape[0], 'Val shape mismatch'
    assert x_test_feat.shape[0] == y_test.shape[0], 'Test shape mismatch'

    assert not np.isnan(x_train_feat).any(), 'NaNs in x_train_feat'
    assert not np.isnan(x_val_feat).any(), 'NaNs in x_val_feat'
    assert not np.isnan(x_test_feat).any(), 'NaNs in x_test_feat'

    assert set(np.unique(y_train)).issubset({cfg.class_a, cfg.class_b})
    assert set(np.unique(y_val)).issubset({cfg.class_a, cfg.class_b})
    assert set(np.unique(y_test)).issubset({cfg.class_a, cfg.class_b})

    print('All data tests passed.')

run_data_tests()

## Section 7 - Training and Evaluation Runner
This section executes only when a model adapter is provided.

In [ ]:
results = {}

if MODEL_ADAPTER is None:
    print('Skipped training: set MODEL_ADAPTER first.')
else:
    t1 = perf_counter()
    MODEL_ADAPTER.fit(x_train_feat, y_train)
    train_seconds = perf_counter() - t1

    y_val_pred = MODEL_ADAPTER.predict(x_val_feat)
    y_test_pred = MODEL_ADAPTER.predict(x_test_feat)

    val_metrics = evaluate_predictions(y_val, y_val_pred, cfg.class_a, cfg.class_b)
    test_metrics = evaluate_predictions(y_test, y_test_pred, cfg.class_a, cfg.class_b)

    results = {
        'feature_mode': cfg.feature_mode,
        'class_a': cfg.class_a,
        'class_b': cfg.class_b,
        'train_samples': int(len(y_train)),
        'val_samples': int(len(y_val)),
        'test_samples': int(len(y_test)),
        'train_seconds': float(train_seconds),
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
    }

    y_test_score = MODEL_ADAPTER.predict_proba(x_test_feat)
    if y_test_score is not None:
        prob_metrics = evaluate_probabilities(y_test, y_test_score, positive_label=cfg.class_b)
        results['test_prob_metrics'] = prob_metrics

    print('Validation metrics:', {k: v for k, v in val_metrics.items() if k != 'confusion_matrix'})
    print('Test metrics:', {k: v for k, v in test_metrics.items() if k != 'confusion_matrix'})

In [ ]:
if results:
    cm = results['test_metrics']['confusion_matrix']

    plt.figure(figsize=(5, 4))
    plt.imshow(cm, cmap='Blues')
    plt.title('Test Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks([0, 1], [cfg.class_a, cfg.class_b])
    plt.yticks([0, 1], [cfg.class_a, cfg.class_b])

    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center')

    plt.colorbar()
    plt.tight_layout()
    plt.show()

    if 'test_prob_metrics' in results:
        fpr = results['test_prob_metrics']['fpr']
        tpr = results['test_prob_metrics']['tpr']
        roc_auc = results['test_prob_metrics']['roc_auc']

        plt.figure(figsize=(5, 4))
        plt.plot(fpr, tpr, label=f'ROC AUC = {roc_auc:.4f}')
        plt.plot([0, 1], [0, 1], '--')
        plt.xlabel('FPR')
        plt.ylabel('TPR')
        plt.title('ROC Curve')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## Section 8 - Artifact Export
Save a concise JSON run report for reproducibility.

In [ ]:
if results:
    export = {
        'config': {
            'random_state': cfg.random_state,
            'class_a': cfg.class_a,
            'class_b': cfg.class_b,
            'test_size': cfg.test_size,
            'val_size': cfg.val_size,
            'feature_mode': cfg.feature_mode,
            'pca_components': cfg.pca_components,
        },
        'class_distribution': {
            'all': dist_all,
            'train': dist_train,
            'val': dist_val,
            'test': dist_test,
        },
        'feature_info': feature_info,
        'results': {
            'feature_mode': results['feature_mode'],
            'class_a': results['class_a'],
            'class_b': results['class_b'],
            'train_samples': results['train_samples'],
            'val_samples': results['val_samples'],
            'test_samples': results['test_samples'],
            'train_seconds': results['train_seconds'],
            'val_metrics': {
                'accuracy': results['val_metrics']['accuracy'],
                'precision_macro': results['val_metrics']['precision_macro'],
                'recall_macro': results['val_metrics']['recall_macro'],
                'f1_macro': results['val_metrics']['f1_macro'],
                'confusion_matrix': results['val_metrics']['confusion_matrix'].tolist(),
            },
            'test_metrics': {
                'accuracy': results['test_metrics']['accuracy'],
                'precision_macro': results['test_metrics']['precision_macro'],
                'recall_macro': results['test_metrics']['recall_macro'],
                'f1_macro': results['test_metrics']['f1_macro'],
                'confusion_matrix': results['test_metrics']['confusion_matrix'].tolist(),
            },
        },
    }

    if 'test_prob_metrics' in results:
        export['results']['test_prob_metrics'] = {
            'roc_auc': results['test_prob_metrics']['roc_auc'],
            'avg_precision': results['test_prob_metrics']['avg_precision'],
        }

    out_path = cfg.output_dir / 'run_report.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(export, f, indent=2)

    print('Saved:', out_path)
else:
    print('No results to export. Set MODEL_ADAPTER and run training first.')

## Section 9 - Next Action
To use this pipeline now:
1. Implement `MODEL_ADAPTER` in Section 4
2. Re-run from Section 5 onward
3. Check metrics, plots, and exported report